# Mission 3 ver3 - 논문 기반 KLUE-RoBERTa 단일 Colab 노트북

이 파일 **하나만 Colab에 올린 뒤 위에서부터 순서대로 실행**하면 됩니다.

## 논문에서 가져온 핵심

이 노트북은 *A Methodology for Emergency Calls Severity Prediction: From Pre-processing to BERT-Based Classifiers* (Abi Kanaan et al., 2023, DOI: `10.1007/978-3-031-34111-3_28`)의 텍스트 모델을 Mission 3에 맞게 적용합니다.

- 논문: 언어별 RoBERTa 계열인 CamemBERT-base를 통화 전사문에 미세조정
- 이 노트북: 한국어 전용 RoBERTa 계열인 `klue/roberta-base`를 사용
- 논문의 caller-operator 설정을 따라 최대 토큰 길이 384, 학습률 `3e-5`, Adam epsilon `1e-7`, 배치 크기 16 사용
- 논문의 이진 severity 출력은 Mission 3의 9개 증상 동시 예측에 맞춰 9개 sigmoid 출력으로 변경

따라서 **논문의 방법을 한국어 다중 레이블 문제로 옮긴 구현**이며, 프랑스어 이진분류 실험을 그대로 재현하는 파일은 아닙니다.

## 출제 규칙

- 학습은 서울 Training만 사용합니다.
- 모델 입력은 JSON의 `utterances[].text`만 사용합니다.
- `symptom`은 Training 학습 정답 또는 최종 평가 정답으로만 사용합니다.
- Validation은 학습, epoch 선택, 임계값 조정에 사용하지 않습니다.
- 9개 외 symptom만 제외하고 통화 샘플은 제거하지 않습니다.

> 이 모델은 GPU가 필요합니다. Colab 런타임에서 A100 또는 L4 GPU를 선택하세요.

In [ ]:
# 1. 필요 패키지 설치
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "transformers": "transformers>=4.45,<5",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("설치 완료:", missing)
else:
    print("필요 패키지가 이미 설치되어 있습니다.")

In [ ]:
# 2. Google Drive 연결 - 모델과 결과를 Drive에 영구 저장합니다.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OUTPUT_DIR = Path("/content/drive/MyDrive/DCC_AIdeas/mission3_ver3_outputs")
except ImportError:
    OUTPUT_DIR = Path.cwd() / "mission3_ver3_outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "mission3_ver3_model.ckpt"
TOKENIZER_DIR = OUTPUT_DIR / "mission3_ver3_tokenizer"
PREDICTION_PATH = OUTPUT_DIR / "mission3.csv"
METRICS_PATH = OUTPUT_DIR / "mission3_ver3_metrics.json"
HISTORY_PATH = OUTPUT_DIR / "mission3_ver3_history.json"

print("영구 저장 폴더:", OUTPUT_DIR)
print("모델 저장 경로:", CHECKPOINT_PATH)

## 3. Google Drive 또는 `/content`에 있는 데이터 찾기

Google Drive의 `MyDrive`를 먼저 확인하고, Drive를 사용할 수 없으면 `/content`에서 `TL_서울_구급`과 `VL_서울_구급`을 찾습니다. 데이터는 ZIP이 아니라 폴더로 압축 해제되어 있어야 합니다.

In [ ]:
# 3. Training/Validation 라벨 폴더 자동 탐색
import os

DRIVE_ROOT = Path("/content/drive/MyDrive")
CONTENT_ROOT = DRIVE_ROOT if DRIVE_ROOT.is_dir() else (
    Path("/content") if Path("/content").is_dir() else Path.cwd()
)
SKIP_DIRS = {"sample_data", ".config", "__pycache__", ".git"}
TARGET_FOLDER_NAMES = {"TL_서울_구급", "VL_서울_구급"}

common_parents = [
    CONTENT_ROOT / "mission3_data",
    CONTENT_ROOT / "대학부 데이터",
    CONTENT_ROOT / "DATA" / "대학부 데이터",
    CONTENT_ROOT / "DCC_AIdeas" / "DATA" / "대학부 데이터",
]
direct_candidates = {
    "TL_서울_구급": [
        CONTENT_ROOT / "mission3_data" / "train" / "TL_서울_구급",
        *[parent / "Training" / "2.라벨링데이터" / "TL_서울_구급" for parent in common_parents],
    ],
    "VL_서울_구급": [
        CONTENT_ROOT / "mission3_data" / "validation" / "VL_서울_구급",
        *[parent / "Validation" / "2.라벨링데이터" / "VL_서울_구급" for parent in common_parents],
    ],
}

located = {}
for folder_name, candidates in direct_candidates.items():
    located[folder_name] = next((path for path in candidates if path.is_dir()), None)

missing_names = {name for name, path in located.items() if path is None}
if missing_names:
    print("일반 경로에서 못 찾은 폴더만 추가 탐색합니다:", missing_names)
    for current, directories, _ in os.walk(CONTENT_ROOT):
        directories[:] = [name for name in directories if name not in SKIP_DIRS]
        current_path = Path(current)
        if current_path.name in missing_names:
            located[current_path.name] = current_path
            directories[:] = []
            missing_names.remove(current_path.name)
            if not missing_names:
                break

if missing_names:
    raise FileNotFoundError(
        f"{sorted(missing_names)} 폴더를 {CONTENT_ROOT}에서 찾지 못했습니다. "
        "압축을 풀었는지와 폴더 이름을 확인하세요."
    )

TRAIN_LABEL_DIR = located["TL_서울_구급"]
VALID_LABEL_DIR = located["VL_서울_구급"]

print("Training JSON 폴더:", TRAIN_LABEL_DIR)
print("Validation JSON 폴더:", VALID_LABEL_DIR)
print("Training JSON 개수:", len(list(TRAIN_LABEL_DIR.rglob("*.json"))))
print("Validation JSON 개수:", len(list(VALID_LABEL_DIR.rglob("*.json"))))

In [ ]:
# 4. 데이터 처리 함수 - 모델 입력으로는 text만 사용합니다.
import csv
import json
import random
import time
from typing import Any

import numpy as np
import torch
from sklearn.metrics import f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

TARGET_SYMPTOMS = (
    "고열", "구토", "두통", "복통", "어지러움",
    "열상", "오심", "전신쇠약", "호흡곤란",
)
UTTERANCE_KEYS = ("utterances", "utterences")

def find_label_files(label_dir: Path) -> list[Path]:
    files = sorted(label_dir.rglob("*.json"))
    if not files:
        raise ValueError(f"JSON 파일이 없습니다: {label_dir}")
    return files

def load_annotation(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as file:
        annotation = json.load(file)
    if not isinstance(annotation, dict):
        raise ValueError(f"JSON object가 아닙니다: {path}")
    return annotation

def dialogue_text(annotation: dict[str, Any]) -> str:
    utterances = []
    for key in UTTERANCE_KEYS:
        if isinstance(annotation.get(key), list):
            utterances = annotation[key]
            break
    texts = []
    for utterance in utterances:
        if not isinstance(utterance, dict):
            continue
        text = utterance.get("text")
        if isinstance(text, str) and text.strip():
            texts.append(text.strip())
    return "\n".join(texts)

def target_symptoms(annotation: dict[str, Any]) -> list[str]:
    raw = annotation.get("symptom", [])
    if raw is None:
        raw = []
    elif isinstance(raw, str):
        raw = [raw]
    if not isinstance(raw, list):
        raise ValueError("symptom은 문자열 list여야 합니다.")
    found = {str(value).strip() for value in raw}
    return [symptom for symptom in TARGET_SYMPTOMS if symptom in found]

def target_vector(annotation: dict[str, Any]) -> list[int]:
    found = set(target_symptoms(annotation))
    return [int(symptom in found) for symptom in TARGET_SYMPTOMS]

def load_dataset(label_dir: Path, include_targets: bool):
    paths = find_label_files(label_dir)
    texts, targets = [], []
    for index, path in enumerate(paths, 1):
        annotation = load_annotation(path)
        texts.append(dialogue_text(annotation))
        if include_targets:
            targets.append(target_vector(annotation))
        if index % 5000 == 0:
            print(f"  {index:,}/{len(paths):,} JSON 읽음")
    target_array = np.asarray(targets, dtype=np.float32) if include_targets else None
    return paths, texts, target_array

print("데이터 처리 함수 준비 완료")

In [ ]:
# 5. Training 데이터를 읽고 9개 증상 분포를 확인합니다.
training_paths, training_texts, training_targets = load_dataset(
    TRAIN_LABEL_DIR, include_targets=True
)

print("\nTraining 샘플:", len(training_paths))
print("빈 대화 텍스트:", sum(not text.strip() for text in training_texts))
print("증상별 정답 개수:")
for symptom, count in zip(TARGET_SYMPTOMS, training_targets.sum(axis=0)):
    print(f"  {symptom:<8} {int(count):>6,}")

print("\n첫 대화 미리보기:")
print(training_texts[0][:500])

## 6. 논문 모델을 Mission 3에 적용하는 방식

논문은 프랑스어 통화에 CamemBERT-base를 사용했습니다. 한국어 문장에 프랑스어 토크나이저를 사용하면 논문의 핵심인 **언어에 맞는 BERT 계열 모델**이라는 장점이 사라지므로 `klue/roberta-base`를 사용합니다.

모델은 첫 토큰의 문맥 표현을 dropout과 하나의 선형 분류층에 통과시켜 9개 점수를 만듭니다. 각 점수에는 sigmoid를 적용하며, 한 통화에서 여러 증상을 동시에 고를 수 있습니다. 논문은 클래스 균형을 위해 데이터를 제거했지만 Mission 3에서는 샘플을 제거하면 안 되므로, 같은 목적을 양성 클래스 가중치가 있는 BCE 손실로 구현합니다.

논문의 15 epoch를 최대값으로 사용하되, 최적 epoch와 증상별 임계값은 오직 Training 내부 holdout으로 결정합니다. 이후 Validation을 보지 않고 Training 전체를 최적 epoch만큼 다시 학습합니다.

In [ ]:
# 6. 논문 기반 학습 설정과 모델 함수
SEED = 42
TEST_SIZE = 0.20
MODEL_NAME = "klue/roberta-base"
MAX_LENGTH = 384
BATCH_SIZE = 16
INFERENCE_BATCH_SIZE = 32
MAX_EPOCHS = 15
LEARNING_RATE = 3e-5
ADAM_EPSILON = 1e-7
DROPOUT = 0.1
NUM_WORKERS = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
if DEVICE.type != "cuda":
    raise RuntimeError(
        "GPU를 찾지 못했습니다. Colab 메뉴의 런타임 > 런타임 유형 변경에서 "
        "A100 또는 L4 GPU를 선택한 뒤 처음부터 다시 실행하세요."
    )

torch.set_float32_matmul_precision("high")

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

class EmergencyTextDataset(Dataset):
    def __init__(self, texts, targets=None):
        self.texts = list(texts)
        self.targets = None if targets is None else np.asarray(targets, dtype=np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        if self.targets is None:
            return self.texts[index]
        return self.texts[index], self.targets[index]

class TokenizeBatch:
    def __init__(self, tokenizer, max_length: int):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, items):
        has_targets = isinstance(items[0], tuple)
        texts = [item[0] for item in items] if has_targets else items
        batch = self.tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        if has_targets:
            batch["labels"] = torch.tensor(
                np.stack([item[1] for item in items]), dtype=torch.float32
            )
        return batch

class KoreanRoBERTaMultiLabel(nn.Module):
    def __init__(self, model_name: str, num_labels: int, dropout: float):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        first_token = outputs.last_hidden_state[:, 0]
        return self.classifier(self.dropout(first_token))

def make_loader(texts, tokenizer, targets=None, batch_size=16, shuffle=False):
    dataset = EmergencyTextDataset(texts, targets)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=TokenizeBatch(tokenizer, MAX_LENGTH),
    )

def make_pos_weight(targets: np.ndarray) -> torch.Tensor:
    positives = targets.sum(axis=0)
    negatives = len(targets) - positives
    weights = negatives / np.maximum(positives, 1.0)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def train_one_epoch(model, loader, optimizer, loss_function, scaler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        labels = batch.pop("labels").to(DEVICE, non_blocking=True)
        inputs = {name: value.to(DEVICE, non_blocking=True) for name, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(**inputs)
            loss = loss_function(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * labels.size(0)
    return total_loss / len(loader.dataset)

def predict_probabilities(model, loader):
    model.eval()
    probabilities = []
    with torch.inference_mode():
        for batch in loader:
            batch.pop("labels", None)
            inputs = {name: value.to(DEVICE, non_blocking=True) for name, value in batch.items()}
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(**inputs)
            probabilities.append(torch.sigmoid(logits).float().cpu().numpy())
    return np.concatenate(probabilities, axis=0)

def tune_thresholds(truth: np.ndarray, probabilities: np.ndarray) -> np.ndarray:
    thresholds = np.full(len(TARGET_SYMPTOMS), 0.5, dtype=np.float32)
    candidates = np.arange(0.10, 0.91, 0.05)
    for column in range(len(TARGET_SYMPTOMS)):
        best_score = -1.0
        for threshold in candidates:
            predicted = (probabilities[:, column] >= threshold).astype(np.int8)
            score = f1_score(truth[:, column], predicted, zero_division=0)
            if score > best_score:
                best_score = score
                thresholds[column] = threshold
    return thresholds

def symptom_scores(truth, probabilities, thresholds):
    predictions = (probabilities >= thresholds).astype(np.int8)
    scores = f1_score(truth, predictions, average=None, zero_division=0)
    return predictions, scores

print("장치:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("모델:", MODEL_NAME)
print("최대 토큰 길이:", MAX_LENGTH)
print("최대 epoch:", MAX_EPOCHS)

In [ ]:
# 7. Training 내부 holdout으로 최적 epoch/임계값을 정하고 Training 전체로 최종 학습
indices = np.arange(len(training_texts))
train_indices, holdout_indices = train_test_split(
    indices, test_size=TEST_SIZE, random_state=SEED, shuffle=True
)
train_texts = [training_texts[index] for index in train_indices]
holdout_texts = [training_texts[index] for index in holdout_indices]
train_targets = training_targets[train_indices]
holdout_targets = training_targets[holdout_indices]

print("KLUE-RoBERTa 토크나이저 다운로드/불러오기...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
train_loader = make_loader(
    train_texts, tokenizer, train_targets, batch_size=BATCH_SIZE, shuffle=True
)
holdout_loader = make_loader(
    holdout_texts, tokenizer, batch_size=INFERENCE_BATCH_SIZE, shuffle=False
)

print("1/2 Training 내부 holdout 학습 시작...")
seed_everything(SEED)
holdout_model = KoreanRoBERTaMultiLabel(
    MODEL_NAME, len(TARGET_SYMPTOMS), DROPOUT
).to(DEVICE)
holdout_optimizer = torch.optim.Adam(
    holdout_model.parameters(), lr=LEARNING_RATE, eps=ADAM_EPSILON
)
holdout_loss = nn.BCEWithLogitsLoss(pos_weight=make_pos_weight(train_targets))
holdout_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

best_macro_f1 = -1.0
best_epoch = 1
best_thresholds = np.full(len(TARGET_SYMPTOMS), 0.5, dtype=np.float32)
best_holdout_scores = np.zeros(len(TARGET_SYMPTOMS), dtype=np.float32)
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    started_at = time.time()
    average_loss = train_one_epoch(
        holdout_model, train_loader, holdout_optimizer, holdout_loss, holdout_scaler
    )
    holdout_probabilities = predict_probabilities(holdout_model, holdout_loader)
    epoch_thresholds = tune_thresholds(holdout_targets, holdout_probabilities)
    _, epoch_scores = symptom_scores(
        holdout_targets, holdout_probabilities, epoch_thresholds
    )
    macro_f1 = float(epoch_scores.mean())
    elapsed_minutes = (time.time() - started_at) / 60
    history.append({
        "epoch": epoch,
        "train_loss": float(average_loss),
        "holdout_macro_f1": macro_f1,
        "minutes": elapsed_minutes,
    })
    print(
        f"epoch {epoch:02d}/{MAX_EPOCHS} | loss={average_loss:.4f} | "
        f"holdout Macro F1={macro_f1:.4f} | {elapsed_minutes:.1f}분"
    )
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_epoch = epoch
        best_thresholds = epoch_thresholds.copy()
        best_holdout_scores = epoch_scores.copy()

print("\nTraining 내부 최적 결과")
print("-" * 60)
for symptom, score, threshold in zip(
    TARGET_SYMPTOMS, best_holdout_scores, best_thresholds
):
    print(f"{symptom:<8} F1={score:.4f}  threshold={threshold:.2f}")
print("-" * 60)
print(f"best epoch={best_epoch}, Macro F1={best_macro_f1:.4f}")

# Holdout 모델을 제거하여 최종 학습용 GPU 메모리를 확보합니다.
del holdout_model, holdout_optimizer, holdout_scaler, holdout_loss
torch.cuda.empty_cache()

print(f"\n2/2 Training 전체를 {best_epoch} epoch 동안 최종 학습합니다...")
seed_everything(SEED)
full_loader = make_loader(
    training_texts, tokenizer, training_targets, batch_size=BATCH_SIZE, shuffle=True
)
final_model = KoreanRoBERTaMultiLabel(
    MODEL_NAME, len(TARGET_SYMPTOMS), DROPOUT
).to(DEVICE)
final_optimizer = torch.optim.Adam(
    final_model.parameters(), lr=LEARNING_RATE, eps=ADAM_EPSILON
)
final_loss = nn.BCEWithLogitsLoss(pos_weight=make_pos_weight(training_targets))
final_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

for epoch in range(1, best_epoch + 1):
    started_at = time.time()
    average_loss = train_one_epoch(
        final_model, full_loader, final_optimizer, final_loss, final_scaler
    )
    print(
        f"final epoch {epoch:02d}/{best_epoch} | loss={average_loss:.4f} | "
        f"{(time.time() - started_at) / 60:.1f}분"
    )

tokenizer.save_pretrained(TOKENIZER_DIR)
final_model.to("cpu")
checkpoint = {
    "model_state_dict": final_model.state_dict(),
    "model_name": MODEL_NAME,
    "model_description": "KLUE-RoBERTa-base + single linear 9-label sigmoid head",
    "target_symptoms": TARGET_SYMPTOMS,
    "thresholds": best_thresholds.tolist(),
    "max_length": MAX_LENGTH,
    "dropout": DROPOUT,
    "best_epoch": best_epoch,
    "training_samples": len(training_texts),
    "seed": SEED,
}
torch.save(checkpoint, CHECKPOINT_PATH)
with HISTORY_PATH.open("w", encoding="utf-8") as file:
    json.dump(history, file, ensure_ascii=False, indent=2)

print("최종 모델 저장 완료:", CHECKPOINT_PATH)
print("토크나이저 저장 완료:", TOKENIZER_DIR)
print("학습 기록 저장 완료:", HISTORY_PATH)

# 저장 후 GPU 메모리를 비워 다음 추론 셀에서 체크포인트를 다시 불러옵니다.
del final_model, final_optimizer, final_scaler, final_loss
torch.cuda.empty_cache()

In [ ]:
# 8. Validation에서는 오직 text만 읽어 예측 CSV를 만듭니다.
validation_paths, validation_texts, _ = load_dataset(
    VALID_LABEL_DIR, include_targets=False
)

saved_checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
saved_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, use_fast=True)
saved_model = KoreanRoBERTaMultiLabel(
    saved_checkpoint["model_name"],
    len(saved_checkpoint["target_symptoms"]),
    saved_checkpoint["dropout"],
)
saved_model.load_state_dict(saved_checkpoint["model_state_dict"])
saved_model.to(DEVICE)

validation_loader = make_loader(
    validation_texts,
    saved_tokenizer,
    batch_size=INFERENCE_BATCH_SIZE,
    shuffle=False,
)
validation_probabilities = predict_probabilities(saved_model, validation_loader)
saved_thresholds = np.asarray(saved_checkpoint["thresholds"], dtype=np.float32)
validation_predictions = (validation_probabilities >= saved_thresholds).astype(np.int8)

rows = []
for path, flags in zip(validation_paths, validation_predictions):
    symptoms = [
        symptom for symptom, present in zip(TARGET_SYMPTOMS, flags) if present
    ]
    rows.append((path.name, str(symptoms)))

with PREDICTION_PATH.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.writer(file)
    writer.writerow(["label file name", "symptom"])
    writer.writerows(rows)

print(f"{len(rows):,}개 예측 CSV 저장 완료: {PREDICTION_PATH}")
print("예측 예시:", rows[:5])

In [ ]:
# 9. Validation은 학습에 넣지 않고 여기서 최종 Macro F1만 계산합니다.
_, _, validation_targets = load_dataset(VALID_LABEL_DIR, include_targets=True)
validation_scores = f1_score(
    validation_targets, validation_predictions, average=None, zero_division=0
)
precision, recall, _, support = precision_recall_fscore_support(
    validation_targets, validation_predictions, average=None, zero_division=0
)
validation_macro_f1 = float(validation_scores.mean())

print("Validation 결과")
print("-" * 80)
print(f"{'증상':<8} {'Precision':>10} {'Recall':>10} {'F1':>8} {'정답수':>8}")
print("-" * 80)
for symptom, p, r, score, count in zip(
    TARGET_SYMPTOMS, precision, recall, validation_scores, support
):
    print(f"{symptom:<8} {p:>10.4f} {r:>10.4f} {score:>8.4f} {int(count):>8}")
print("-" * 80)
print(f"Validation Macro F1 = {validation_macro_f1:.4f}")

metrics = {
    "paper": "A Methodology for Emergency Calls Severity Prediction: From Pre-processing to BERT-Based Classifiers",
    "doi": "10.1007/978-3-031-34111-3_28",
    "model": saved_checkpoint["model_description"],
    "training_samples": len(training_texts),
    "validation_samples": len(validation_texts),
    "best_epoch": int(saved_checkpoint["best_epoch"]),
    "training_holdout_macro_f1": float(best_macro_f1),
    "validation_macro_f1": validation_macro_f1,
    "per_symptom_f1": {
        symptom: float(score) for symptom, score in zip(TARGET_SYMPTOMS, validation_scores)
    },
    "thresholds": {
        symptom: float(value) for symptom, value in zip(TARGET_SYMPTOMS, saved_thresholds)
    },
}
with METRICS_PATH.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
print("점수 기록 저장 완료:", METRICS_PATH)

## 10. 결과 파일

모든 결과는 Google Drive의 다음 폴더에 저장됩니다.

```text
MyDrive/DCC_AIdeas/mission3_ver3_outputs/
├─ mission3_ver3_model.ckpt
├─ mission3_ver3_tokenizer/
├─ mission3.csv
├─ mission3_ver3_metrics.json
└─ mission3_ver3_history.json
```

- `.ckpt`: KLUE-RoBERTa와 9개 증상 분류층의 학습 가중치
- `tokenizer/`: 한국어 문장을 모델 입력 숫자로 바꾸는 파일
- `mission3.csv`: 제출 형식의 Validation 예측
- `metrics.json`: Macro F1, 증상별 F1, threshold
- `history.json`: epoch별 Training 내부 점수

기존 TF-IDF용 `inference.py`는 이 체크포인트를 읽을 수 없습니다. 최종 제출 전에 ver3 전용 추론 스크립트가 별도로 필요합니다.

In [ ]:
# 10. 원할 때만 True로 바꾸어 CSV를 PC로 다운로드하세요.
DOWNLOAD_CSV = False

if DOWNLOAD_CSV:
    try:
        from google.colab import files
        files.download(str(PREDICTION_PATH))
    except ImportError:
        print("Colab에서만 자동 다운로드할 수 있습니다.")
else:
    print("CSV는 Drive에 저장되었습니다:", PREDICTION_PATH)

## 11. 실험 메모

실행이 끝나면 아래를 채우고 **노트북을 저장**하세요. 셀 출력이 학습 로그가 됩니다.

- 실험 날짜:
- GPU:
- 최적 epoch:
- Training 내부 Macro F1:
- Validation Macro F1:
- 가장 잘 맞힌 증상:
- 가장 못 맞힌 증상:

> Validation 결과를 보며 epoch, threshold, 모델 설정을 반복해서 바꾸지 마세요. 모델 선택은 Training 내부 holdout 결과만 사용해야 합니다.